In [21]:
#!pip install gensim
import gensim.downloader as api
import numpy as np
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [22]:
glove_model = api.load('glove-wiki-gigaword-100')
print(f'Glove Vocab size : {len(glove_model)}')

Glove Vocab size : 400000


In [23]:
df = pd.read_csv('/content/IMDB Dataset.csv')
df = df[:20000]

In [24]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [44]:
df.shape

(20000, 3)

In [25]:
df['sentiment'].value_counts()

,count
sentiment,
negative,10097
positive,9903


In [26]:
nltk.download('punkt_tab')
def Processing_Pipeline(text):
    # 1. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 2. Lowercasing
    text = text.lower()
    # 3. Remove punctuation
    text = re.sub(r'[^​\w\s]', ' ', text)
    # 4. Tokenization
    tokens = word_tokenize(text)
    # 5. Join back
    return " ".join(tokens)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [27]:
df['clean_review'] = df['review'].apply(Processing_Pipeline)

In [29]:
X = df['clean_review']
y = df['sentiment']
y = y.map({'positive': 1, 'negative': 0})

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
def get_sent_vector(text):
  vectors = []

  for word in text.split():
    if word in glove_model:
      vectors.append(glove_model[word])

  if len(vectors) == 0:
    return np.zeros

  return np.mean(vectors, axis=0)

In [37]:
X_train_vec = np.array([get_sent_vector(text) for text in X_train])
X_test_vec = np.array([get_sent_vector(text) for text in X_test])

In [45]:
X_train_vec.shape

(16000, 100)

In [48]:


model = RandomForestClassifier()
model.fit(X_train_vec, y_train)

RandomForestClassifier()

In [49]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.762


In [51]:
print("\nFull Report:")
print(classification_report(y_test, y_pred))


Full Report:
              precision    recall  f1-score   support

           0       0.78      0.75      0.76      2048
           1       0.75      0.77      0.76      1952

    accuracy                           0.76      4000
   macro avg       0.76      0.76      0.76      4000
weighted avg       0.76      0.76      0.76      4000

